In [1]:
%pip install -qU langchain langchain-ollama langgraph pydanti

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement pydanti (from versions: none)

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for pydanti


In [2]:
import os
import sys
import json
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

Path config

In [3]:

DIRETORIO_ATUAL = os.getcwd()
DIRETORIO_RAIZ = os.path.dirname(DIRETORIO_ATUAL)

sys.path.append(DIRETORIO_RAIZ)
print(f"Current Directory: {DIRETORIO_ATUAL}")
print(f"Root Directory added to path: {DIRETORIO_RAIZ}")

Current Directory: e:\FIAP\prompt\Sprint-3\notebooks
Root Directory added to path: e:\FIAP\prompt\Sprint-3


Prompts e Evals

In [4]:
caminho_prompt = os.path.join(DIRETORIO_RAIZ, "prompts", "system_prompt.md")

try:
    with open(caminho_prompt, "r", encoding="utf-8") as arquivo:
        SYSTEM_PROMPT = arquivo.read()
    print("system_prompt.md successfully loaded.")
except FileNotFoundError:
    print(f"Error: File not found. Path attempted: '{caminho_prompt}'")
    SYSTEM_PROMPT = "Você é um assistente médico virtual da Care Plus."

caminho_evals = os.path.join(DIRETORIO_RAIZ, "evals", "sprint1_eval_set.json")

try:
    with open(caminho_evals, "r", encoding="utf-8") as arquivo:
        eval_cases = json.load(arquivo)
    print("Evals successfully loaded.")
    
    # mostrar evals carregados
    for caso in eval_cases:
        print(f"[TEST: {caso.get('id', 'N/A')} | CATEGORY: {caso.get('categoria', 'N/A').upper()}]")
except FileNotFoundError:
    print(f"Error: File not found. Path attempted: '{caminho_evals}'")
    eval_cases = []

system_prompt.md successfully loaded.
Evals successfully loaded.
[TEST: eval_001 | CATEGORY: HAPPY_PATH]
[TEST: eval_002 | CATEGORY: HAPPY_PATH]
[TEST: eval_003 | CATEGORY: HAPPY_PATH]
[TEST: eval_004 | CATEGORY: RED_FLAG]
[TEST: eval_005 | CATEGORY: RED_FLAG]
[TEST: eval_006 | CATEGORY: RED_FLAG]
[TEST: eval_007 | CATEGORY: JAILBREAK]
[TEST: eval_008 | CATEGORY: JAILBREAK]
[TEST: eval_009 | CATEGORY: OUT_OF_SCOPE]
[TEST: eval_010 | CATEGORY: OUT_OF_SCOPE]
[TEST: eval_sec_011 | CATEGORY: JAILBREAK]
[TEST: eval_sec_012 | CATEGORY: JAILBREAK]
[TEST: eval_sec_013 | CATEGORY: RED_FLAG]
[TEST: eval_sec_014 | CATEGORY: JAILBREAK]
[TEST: eval_sec_015 | CATEGORY: JAILBREAK]


Tools

In [5]:
try:
    from tools.tools_spec import tools
    print("Tools successfully loaded.")
except ImportError as e:
    print(f"Error: Could not import tools. Detail: {e}")
    tools = []

Tools successfully loaded.


LangGraph LLM

In [6]:

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOllama(model="llama3.1", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    mensagens_com_sys_prompt = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    resposta = llm_with_tools.invoke(mensagens_com_sys_prompt)
    return {"messages": [resposta]}

Compile Graph

In [7]:
graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools))

# Routing
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition) 
graph_builder.add_edge("tools", "chatbot") 

# MemorySaver
memory = MemorySaver()
app = graph_builder.compile(checkpointer=memory)

print("LangGraph orchestration with memory compiled.")

LangGraph orchestration with memory compiled.


Chat e Demonstração

In [8]:
# chat
def conversar(mensagem, thread_id="sessao_01"):
    print(f"Patient: {mensagem}")
    config = {"configurable": {"thread_id": thread_id}}
    
    for evento in app.stream({"messages": [("user", mensagem)]}, config, stream_mode="values"):
        ultima_mensagem = evento["messages"][-1]
        
    print(f"BluaDiagnostics: {ultima_mensagem.content}\n")
    print("-" * 70 + "\n")

print("=== STARTING BLUADIAGNOSTICS DEMONSTRATION ===\n")

conversar("Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.")
conversar("Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.")
conversar("Entendi. Mudando de assunto, por que o valor do meu plano de saúde aumentou 15% neste mês? Quero contestar a fatura.")

=== STARTING BLUADIAGNOSTICS DEMONSTRATION ===

Patient: Meu relógio marcou que minha frequência cardíaca está em 110 bpm em repouso, mas estou me sentindo normal. Meu ID na Care Plus é 9988.

[TOOL EXECUTED: Fetching wearable data for patient ID 9988...]
BluaDiagnostics: Sua frequência cardíaca está um pouco acima do normal, mas isso pode não ser necessariamente um problema. Gostaria de saber mais sobre seus sintomas atuais e como você se sente. Você tem algum desconforto ou dor em algum lugar do corpo? Além disso, há algo que o tenha feito sentir-se assim hoje?

----------------------------------------------------------------------

Patient: Respondendo à sua pergunta, não tomei café hoje e não passei por estresse. Acabei de acordar.

[TOOL EXECUTED: Fetching wearable data for patient ID 9988...]
BluaDiagnostics: Com base nos dados que você forneceu, sua frequência cardíaca está um pouco elevada, mas isso pode ser normal após o acordar. Alguns fatores podem contribuir para isso, como